# 3. Finding repeated phrases

In [notebook 2](02-getting-your-text-ready.ipynb) we turned our raw transcriptions into `documents` &mdash; a list of tokenized paragraphs. Now we'll let the software discover candidate formulaic phrases on its own, without us telling it in advance what to look for.

We'll reload the same dataset and tokenize it exactly as before, so this notebook can be run on its own.

In [1]:
import csv
import gzip
import re
from collections import Counter

TOKEN_PATTERN = re.compile(r"[A-Za-z\u00c0-\u00ff]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


rows = []
with gzip.open('data/resolution_paragraphs/paragraphs-3823.tsv.gz', 'rt', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        rows.append(row)

documents = [tokenize(row['text']) for row in rows]
print(f'{len(documents)} documents ready')

2909 documents ready


## The core idea

How could a computer possibly recognise a formula it has never been told about? The underlying idea is simple, even if the bookkeeping involved isn't:

> **Two words that keep showing up right next to each other, over and over, across many different paragraphs, are probably part of the same fixed phrase.**

This means the software needs to do two things before it can search for phrases at all:

1. **Count how often each individual word occurs.** Words that occur only once or twice in the whole corpus are usually one-off details &mdash; a name, a place, a typo &mdash; not the fixed wording of a formula. The software can safely ignore these from the start, which also makes everything that follows faster.
2. **Count how often pairs of words occur near each other.** This is the actual evidence for "these two words tend to belong together."

Let's do both, using the toolkit's `FormulaSearch` class.

In [2]:
from formula_detection.search import FormulaSearch

formula_search = FormulaSearch(
    documents,
    min_term_freq=10,   # ignore words that occur fewer than 10 times in total
    min_cooc_freq=10,   # ignore word *pairs* that occur near each other fewer than 10 times
    skip_size=2,        # "near each other" means within 2 words either side
)

1. Iterating over sentences to calculate term frequencies
    full collection size (tokens): 449,905
    full lexicon size (types): 21,185
    minimum term frequency: 10
    minimum frequency lexicon size: 2,880
2. Iterating over sentences to calculate the co-occurrence frequencies


docs: 2,909	num_words: 449,905	num_coocs: 1,125,317	num distinct coocs: 182,121
    co-occurrence index size: 182,121


The output above is the software reporting on its own progress. Two numbers are worth pausing on:

* `full lexicon size (types)` is how many distinct words it found in the whole corpus.
* `minimum frequency lexicon size` is how many of those words occur at least `min_term_freq` (here, 10) times.

Let's print those two numbers directly, so the filtering is visible in plain terms.

In [3]:
print(f'distinct words in total: {len(formula_search.full_vocab)}')
print(f'distinct words occurring at least 10 times: {len(formula_search.min_freq_vocab)}')

distinct words in total: 21185
distinct words occurring at least 10 times: 2880


Out of roughly 21 thousand distinct words, fewer than 3 thousand occur often enough to be worth considering as part of a formula. The other 18 thousand &mdash; mostly names of people and places, which is exactly what we'd expect from a diplomatic record like this one &mdash; are set aside, not because they're unimportant, but because they're the *variable* content the formulas surround, not the formulas themselves (recall the `[NAME]`, `[PLACE]` slots from notebook 1).

`min_cooc_freq=10` did the second job: it counted, for every pair of frequent words, how often they occurred within 2 words of each other anywhere in the corpus, and only kept pairs that happened at least 10 times. That's the raw material the software now has to work with.

## Asking for candidate phrases

With that groundwork done, we can ask `formula_search` to slide a small window across every paragraph &mdash; here, 6 words wide &mdash; and keep any window where every word inside it is frequent enough, and well-connected enough to its neighbours, based on the counting we just did.

Each such window is a **candidate phrase**: something the software thinks *might* be (part of) a formula. We collect every candidate it finds, and count how often each distinct one occurs.

In [4]:
phrase_freq = Counter()
for match in formula_search.extract_phrases('sub_phrases', min_phrase_length=3, max_phrase_length=6,
                                            min_neighbour_cooc=1):
    phrase_freq[match.candidate_phrase.phrase_string] += 1

print(f'{len(phrase_freq)} distinct candidate phrases found')
print()
print('The 12 most frequent:')
for phrase, freq in phrase_freq.most_common(12):
    print(f'  {freq: >4}  {phrase}')

Minimum co-occurrence frequency: 10


95723 distinct candidate phrases found

The 12 most frequent:
   839  op gedelibereert zynde is goedgevonden en
   839  gedelibereert zynde is goedgevonden en verstaan
   836  waar op gedelibereert zynde is goedgevonden
   835  waar op geen resolutie is gevallen
   720  advertentie waar op geen resolutie is
   718  houdende advertentie waar op geen resolutie
   674  zynde is goedgevonden en verstaan dat
   635  maand houdende advertentie waar op geen
   508  ontfangen een missive van den heere
   427  den deeser loopende maand houdende advertentie
   419  loopende maand houdende advertentie waar op
   415  deeser loopende maand houdende advertentie waar


**Second, something that looks strange at first but makes sense once explained:** several of the top results look like slightly shifted versions of each other. In the list above, the top three are:

```
839  op gedelibereert zynde is goedgevonden en
839  gedelibereert zynde is goedgevonden en verstaan
836  waar op gedelibereert zynde is goedgevonden
```

These aren't three different formulas. They're three overlapping 6-word windows over *one* longer formula, each shifted along by a single word. Because we asked for windows of exactly 6 words, a formula that's actually longer gets reported back to us in several overlapping pieces. If you scan further down the same top-12 list, you'll find a fourth piece of the same formula: `zynde is goedgevonden en verstaan dat` (674) &mdash; it ranks lower because the underlying formula sometimes continues differently from this point, so this exact continuation doesn't occur quite as often as the first three. Lining up the repeated words across all four, you can reconstruct the formula by hand:

> waar op gedelibereert zynde is goedgevonden en verstaan dat ...

("whereupon, having been deliberated, it is found and agreed and understood that ...") &mdash; a real formula introducing a decision, here pieced together from its fragments.

This overlapping-fragments pattern is something you'll see every time you do this kind of search, and it's useful to recognise rather than be confused by: **a cluster of similar, highly-ranked candidate phrases that share most of their words is usually evidence of one longer formula, not several short ones.** In [notebook 6](06-naming-your-formulas.ipynb), we'll use an interactive tool that helps with exactly this kind of piecing-together, instead of doing it by eye as we just did.

## Not every candidate is a real formula

One more thing worth keeping in mind: the software has no idea what counts as historically meaningful. It only knows about frequency and co-occurrence. Most of the time, in a corpus this formulaic, the top results genuinely are recognisable formulas &mdash; but you should expect some candidates further down the list to be ordinary, unremarkable word combinations that simply happen to recur a lot (common short connecting phrases, for instance), without being a "formula" in the sense we cared about in notebook 1.

Deciding which candidates are genuinely meaningful formulas, what they mean, and how to group differently-worded versions of the same formula together, is a judgement call that the software can support but not make for you. That's exactly what the next two notebooks are about.

## Next steps

We now have a list of real, automatically-discovered candidate formulas, frequency-ranked, with the most common ones clearly recognisable. But this corpus spans a single year of resolutions written by a fairly consistent set of clerks. Over longer time spans, or across different scribes, the *same* formula is often spelled in several different ways. [Notebook 4](04-old-spelling-same-phrase.ipynb) looks at how to handle that, so that spelling variation doesn't make the software think "the same formula, written two ways" is actually two different, unrelated phrases.